In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv(r"D:\SML_project_folder\DataCoSupplyChainDataset.csv", encoding="ISO-8859-1")
print(df.head())
df.info()
df.describe()

       Type  Days for shipping (real)  Days for shipment (scheduled)  \
0     DEBIT                         3                              4   
1  TRANSFER                         5                              4   
2      CASH                         4                              4   
3     DEBIT                         3                              4   
4   PAYMENT                         2                              4   

   Benefit per order  Sales per customer   Delivery Status  \
0          91.250000          314.640015  Advance shipping   
1        -249.089996          311.359985     Late delivery   
2        -247.779999          309.720001  Shipping on time   
3          22.860001          304.809998  Advance shipping   
4         134.210007          298.250000  Advance shipping   

   Late_delivery_risk  Category Id   Category Name Customer City  ...  \
0                   0           73  Sporting Goods        Caguas  ...   
1                   1           73  Sporting Goo

,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Late_delivery_risk,Category Id,Customer Id,Customer Zipcode,Department Id,Latitude,...,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Price,Product Status
count,180519.000000,180519.000000,180519.000000,180519.000000,180519.000000,180519.000000,180519.000000,180516.000000,180519.000000,180519.000000,...,180519.000000,180519.000000,180519.000000,180519.000000,24840.000000,180519.000000,180519.000000,0.0,180519.000000,180519.0
mean,3.497654,2.931847,21.974989,183.107609,0.548291,31.851451,6691.379495,35921.126914,5.443460,29.719955,...,2.127638,203.772096,183.107609,21.974989,55426.132327,692.509764,31.851451,NaN,141.232550,0.0
std,1.623722,1.374449,104.433526,120.043670,0.497664,15.640064,4162.918106,37542.461122,1.629246,9.813646,...,1.453451,132.273077,120.043670,104.433526,31919.279101,336.446807,15.640064,NaN,139.732492,0.0
min,0.000000,0.000000,-4274.979980,7.490000,0.000000,2.000000,1.000000,603.000000,2.000000,-33.937553,...,1.000000,9.990000,7.490000,-4274.979980,1040.000000,19.000000,2.000000,NaN,9.990000,0.0
25%,2.000000,2.000000,7.000000,104.379997,0.000000,18.000000,3258.500000,725.000000,4.000000,18.265432,...,1.000000,119.980003,104.379997,7.000000,23464.000000,403.000000,18.000000,NaN,50.000000,0.0
50%,3.000000,4.000000,31.520000,163.990005,1.000000,29.000000,6457.000000,19380.000000,5.000000,33.144863,...,1.000000,199.919998,163.990005,31.520000,59405.000000,627.000000,29.000000,NaN,59.990002,0.0
75%,5.000000,4.000000,64.800003,247.399994,1.000000,45.000000,9779.000000,78207.000000,7.000000,39.279617,...,3.000000,299.950012,247.399994,64.800003,90008.000000,1004.000000,45.000000,NaN,199.990005,0.0
max,6.000000,4.000000,911.799988,1939.989990,1.000000,76.000000,20757.000000,99205.000000,12.000000,48.781933,...,5.000000,1999.989990,1939.989990,911.799988,99301.000000,1363.000000,76.000000,NaN,1999.989990,0.0


In [2]:
"""MISSING VALUES & REDUNDANT COLUMNS HANDLING"""
missing_counts = df.isnull().sum()
cols_with_missing_vals = missing_counts[missing_counts > 0]
print(cols_with_missing_vals)
print()

missing_percentage = (cols_with_missing_vals / len(df)) * 100
print(missing_percentage.round(2).astype(str) + "%")
print()

# Drop 100% empty, 86% missing, zero-variance, masked PII, and 100% duplicate columns
cols_to_drop = [
    "Product Description",        # 100% missing
    "Order Zipcode",              # 86.24% missing
    "Product Status",             # zero-variance (all 0)
    "Customer Email",             # masked PII ('XXXXXXXXX')
    "Customer Password",          # masked PII ('XXXXXXXXX')
    "Customer Fname",
    "Customer Lname",
    "Customer Street",
    "Product Image",
    # 100% identical duplicate columns
    "Benefit per order",          # identical to Order Profit Per Order
    "Sales per customer",         # identical to Order Item Total
    "Product Price",              # identical to Order Item Product Price
    "Category Id",                # identical to Product Category Id
    "Product Card Id",            # identical to Order Item Cardprod Id
    "Customer Id",                # identical to Order Customer Id
]
df.drop(columns=[col for col in cols_to_drop if col in df.columns], inplace=True)

# Replace remaining missing numerical values using the median
num_cols = df.select_dtypes(include=["number"]).columns
for col in num_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

# Replace remaining missing categorical values using the mode
cat_cols = df.select_dtypes(include=["object", "category", "string"]).columns
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print(f"Total missing values: {df.isnull().sum().sum()}")
print(f"Shape after column cleanup: {df.shape}")

Customer Lname              8
Customer Zipcode            3
Order Zipcode          155679
Product Description    180519
dtype: int64

Customer Lname           0.0%
Customer Zipcode         0.0%
Order Zipcode          86.24%
Product Description    100.0%
dtype: str

Total missing values: 0
Shape after column cleanup: (180519, 38)


In [3]:
"""DUPLICATE RECORDS HANDLING"""
duplicates = df[df.duplicated()]
print(f"TOTAL DUPLICATE ROWS FOUND: {len(duplicates)}")

if len(duplicates) > 0:
    print("\n--- DUPLICATE RECORDS ---")
    print(duplicates)

# Remove duplicate records in-place
original_shape = df.shape
df.drop_duplicates(inplace=True)

print("\n--- VERIFICATION ---")
print(f"Original shape: {original_shape}")
print(f"Cleaned shape:  {df.shape}")
print(f"Remaining duplicate rows: {df.duplicated().sum()}")

TOTAL DUPLICATE ROWS FOUND: 0

--- VERIFICATION ---
Original shape: (180519, 38)
Cleaned shape:  (180519, 38)
Remaining duplicate rows: 0


In [4]:
"""DATE AND TIME HANDLING"""
df["order_date"] = pd.to_datetime(df["order date (DateOrders)"], errors="coerce")
df["shipping_date"] = pd.to_datetime(df["shipping date (DateOrders)"], errors="coerce")

# Drop original unparsed string date columns
df.drop(
    columns=["order date (DateOrders)", "shipping date (DateOrders)"],
    inplace=True,
)

# Calculate actual shipping duration (in fractional days)
df["real_shipping_days_calc"] = (
    df["shipping_date"] - df["order_date"]
).dt.total_seconds() / (24 * 3600)

# Extract ML-friendly temporal features
df["order_year"] = df["order_date"].dt.year
df["order_month"] = df["order_date"].dt.month
df["order_day"] = df["order_date"].dt.day
df["order_dayofweek"] = df["order_date"].dt.dayofweek  # 0 = Monday, 6 = Sunday
df["order_hour"] = df["order_date"].dt.hour
df["is_weekend"] = df["order_dayofweek"].isin([5, 6]).astype(int)

# Check data integrity
invalid_dates = (df["shipping_date"] < df["order_date"]).sum()
print(f"Invalid date entries (shipping before order): {invalid_dates}")

# Drop raw datetime objects now that numeric features are extracted
df.drop(columns=["order_date", "shipping_date"], inplace=True)

output_file_path = r"D:\SML_project_folder\cleaned_supply_chain.csv"
df.to_csv(output_file_path, index=False)
print(f"Date correction complete! Saved processed data to: {output_file_path}")

Invalid date entries (shipping before order): 0
Date correction complete! Saved processed data to: D:\SML_project_folder\cleaned_supply_chain.csv


In [5]:
"""OUTLIER DETECTION AND TREATMENT"""
skewed_cols = [
    "Sales",
    "Order Item Total",
    "Order Profit Per Order",
    "Order Item Product Price",
]


def detect_outliers_iqr(dataframe, columns):
    print("--- OUTLIER SUMMARY (IQR Method) ---")
    for col in columns:
        Q1 = dataframe[col].quantile(0.25)
        Q3 = dataframe[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        outliers = dataframe[
            (dataframe[col] < lower_bound) | (dataframe[col] > upper_bound)
        ]
        pct = (len(outliers) / len(dataframe)) * 100
        print(f"{col}: {len(outliers)} outliers found ({pct:.2f}%)")


detect_outliers_iqr(df, skewed_cols)


def cap_outliers_percentile(
    dataframe, columns, lower_percentile=0.01, upper_percentile=0.99
):
    df_capped = dataframe.copy()
    for col in columns:
        lower_limit = df_capped[col].quantile(lower_percentile)
        upper_limit = df_capped[col].quantile(upper_percentile)

        df_capped[col] = np.clip(df_capped[col], lower_limit, upper_limit)
        print(f"Capped {col} between [{lower_limit:.2f}, {upper_limit:.2f}]")

    return df_capped


# Apply capping treatment directly back to df so all downstream cells use it
df = cap_outliers_percentile(df, skewed_cols)

# Save updated master dataset (Use this file for EDA, Dashboard & code4_clustering.ipynb)
df.to_csv(r"D:\SML_project_folder\cleaned_supply_chain.csv", index=False)
print("\nOutlier treatment complete. Cleaned master file saved successfully.")

--- OUTLIER SUMMARY (IQR Method) ---
Sales: 488 outliers found (0.27%)
Order Item Total: 1943 outliers found (1.08%)
Order Profit Per Order: 18942 outliers found (10.49%)
Order Item Product Price: 2048 outliers found (1.13%)
Capped Sales between [24.99, 499.95]
Capped Order Item Total between [22.49, 464.95]
Capped Order Profit Per Order between [-415.60, 184.23]
Capped Order Item Product Price between [15.99, 452.04]

Outlier treatment complete. Cleaned master file saved successfully.


In [6]:
"""CATEGORICAL ENCODING"""
df = pd.read_csv(r"D:\SML_project_folder\cleaned_supply_chain.csv")

cat_cols = df.select_dtypes(include=["object", "category", "string"]).columns
print(f"Categorical columns found: {list(cat_cols)}")

columns_to_encode = [
    "Type",
    "Shipping Mode",
    "Market",
    "Order Region",
    "Customer Segment",
    "Department Name",
]

df_encoded = pd.get_dummies(
    df,
    columns=[col for col in columns_to_encode if col in df.columns],
    drop_first=True,
    dtype=int,
)

print(f"Original shape before encoding: {df.shape}")
print(f"New shape after One-Hot Encoding: {df_encoded.shape}")

output_file_path = r"D:\SML_project_folder\encoded_supply_chain.csv"
df_encoded.to_csv(output_file_path, index=False)
print(f"\nCategorical encoding complete! Saved to: {output_file_path}")

Categorical columns found: ['Type', 'Delivery Status', 'Category Name', 'Customer City', 'Customer Country', 'Customer Segment', 'Customer State', 'Department Name', 'Market', 'Order City', 'Order Country', 'Order Region', 'Order State', 'Order Status', 'Product Name', 'Shipping Mode']
Original shape before encoding: (180519, 43)
New shape after One-Hot Encoding: (180519, 81)

Categorical encoding complete! Saved to: D:\SML_project_folder\encoded_supply_chain.csv


In [7]:
"""FEATURE SELECTION, TRAIN-TEST SPLIT & SCALING"""
df = pd.read_csv(r"D:\SML_project_folder\encoded_supply_chain.csv")

drop_cols = [
    # Target leakage (post-shipment variables)
    "Delivery Status",
    "Days for shipping (real)",
    "real_shipping_days_calc",
    "Order Status",
    # Arbitrary IDs & Zipcode
    "Order Id",
    "Order Item Id",
    "Order Customer Id",
    "Order Item Cardprod Id",
    "Product Category Id",
    "Department Id",
    "Customer Zipcode",
    # High-cardinality text columns
    "Customer City",
    "Customer State",
    "Customer Country",
    "Order City",
    "Order State",
    "Order Country",
    "Product Name",
    "Category Name",
]

existing_drops = [col for col in drop_cols if col in df.columns]
df_model = df.drop(columns=existing_drops)

target_column = "Late_delivery_risk"

if target_column in df_model.columns:
    X = df_model.drop(columns=[target_column])
    y = df_model[target_column]
else:
    print(f"Target column '{target_column}' not found. Please check your target name.")
    X = df_model.select_dtypes(include=["number"])
    y = None

# Stratified Train-Test Split
if y is not None:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
else:
    X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)

# Feature Scaling (fit ONLY on X_train)
scaler = StandardScaler()

cols_to_scale = [
    "Days for shipment (scheduled)",
    "Latitude",
    "Longitude",
    "Order Item Discount",
    "Order Item Discount Rate",
    "Order Item Product Price",
    "Order Item Profit Ratio",
    "Order Item Quantity",
    "Sales",
    "Order Item Total",
    "Order Profit Per Order",
    "order_year",
    "order_month",
    "order_day",
    "order_dayofweek",
    "order_hour",
]
existing_scale_cols = [col for col in cols_to_scale if col in X_train.columns]

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[existing_scale_cols] = scaler.fit_transform(
    X_train[existing_scale_cols]
)
X_test_scaled[existing_scale_cols] = scaler.transform(
    X_test[existing_scale_cols]
)

print(f"Training features shape: {X_train_scaled.shape}")
print(f"Testing features shape: {X_test_scaled.shape}")
print(f"Non-numeric columns remaining: {X_train_scaled.select_dtypes(exclude=['number']).columns.tolist()}")
print("\nFeature scaling complete! Full preprocessed data is ready.")


Training features shape: (144415, 61)
Testing features shape: (36104, 61)
Non-numeric columns remaining: []

Feature scaling complete! Full preprocessed data is ready.
